In [6]:
from transformers import pipeline

gen = pipeline("text-generation", model="gpt2")
print(gen("In the year 2050, cities will", max_new_tokens=50)[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In the year 2050, cities will have 10,000 more square kilometers of land to occupy, and the number of square kilometers of farmland will reach 10.5 million.

"The country is not going to be able to build up a lot of urban development without creating a lot


#  HuggingFace `pipeline()` — Quick Notes

---

##  What is `pipeline`?

`pipeline()` is HuggingFace's **easiest way to use a pre-trained model** — one line loads the model, tokenizer, and does all pre/post-processing for you.

>  No manual tokenizing, no tensor handling — just give text in, get results out.

---

## Syntax

```python
from transformers import pipeline
pipe = pipeline(task, model="model_name")
result = pipe(input_text)
```

| Part | Meaning |
|------|---------|
| `task` | What job to do (see below) |
| `model` | Which pre-trained model to use (optional — has defaults) |
| `input_text` | Your text/data |

### Common Tasks
| Task | Does |
|------|------|
| `"text-generation"` | Continues/writes text (GPT-style) |
| `"sentiment-analysis"` | Positive/negative |
| `"summarization"` | Shortens text |
| `"translation"` | Language → language |
| `"question-answering"` | Answers from context |

---

## The Code

```python
from transformers import pipeline

gen = pipeline("text-generation", model="gpt2")
print(gen("In the year 2050, cities will", max_new_tokens=50)[0]["generated_text"])
```

| Part | Meaning |
|------|---------|
| `pipeline("text-generation", model="gpt2")` | Loads **GPT-2** for text generation |
| `gen("In the year 2050...")` | The **prompt** — model continues from here |
| `max_new_tokens=50` | Generate up to **50 new tokens** (words/pieces) |
| `[0]` | Result is a list → grab first output |
| `["generated_text"]` | Extract the actual text string from the result dict |

---

## Output Explained

- **Progress bar** `Loading weights: 100%` → GPT-2 weights downloading/loading (one-time).
- **Warning** `Both max_new_tokens and max_length set` → harmless; `max_new_tokens` wins.
- **Generated text** → GPT-2's continuation of your prompt.

> GPT-2 is **old & small** → output is often rambly/nonsensical. That's expected, not a bug.

---

## Key Takeaways

| Point | Detail |
|-------|--------|
| `pipeline` = 1-line model usage | Handles everything internally |
| Output is a **list of dicts** | Use `[0]["generated_text"]` |
| `max_new_tokens` | Controls output length |
| GPT-2 = weak model | Great for learning, weak results |

In [7]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

enc = tok("Tokenizers split rare words.", return_tensors="pt")
print(enc["input_ids"])
print(tok.convert_ids_to_tokens(enc["input_ids"][0]))
# ['[CLS]', 'token', '##izers', 'split', 'rare', 'words', '.', '[SEP]']

# Batch → needs padding so tensors are rectangular
batch = tok(["short one", "a much longer sentence here"],
            padding=True, truncation=True, return_tensors="pt")
print(batch["input_ids"])
print(batch["attention_mask"])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tensor([[  101, 19204, 17629,  2015,  3975,  4678,  2616,  1012,   102]])
['[CLS]', 'token', '##izer', '##s', 'split', 'rare', 'words', '.', '[SEP]']
tensor([[ 101, 2460, 2028,  102,    0,    0,    0],
        [ 101, 1037, 2172, 2936, 6251, 2182,  102]])
tensor([[1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1]])


# AutoTokenizer — Converting Text to Model-Ready Numbers

---

## What is a Tokenizer? (Concept)

A **tokenizer** converts text into **numbers (token IDs)** that a transformer model can process. It also splits rare/unknown words into smaller sub-word pieces.

> `AutoTokenizer` auto-loads the **correct tokenizer** for whatever model you name — no need to know the exact tokenizer class.

---

## Syntax

```python
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("model-name")
enc = tok("your text", return_tensors="pt")
```

| Part | Meaning |
|------|---------|
| `from_pretrained("bert-base-uncased")` | Downloads & loads BERT's tokenizer |
| `return_tensors="pt"` | Return **PyTorch** tensors (`"tf"` for TensorFlow) |

---

## The Code — Part 1: Single Sentence

```python
enc = tok("Tokenizers split rare words.", return_tensors="pt")
print(enc["input_ids"])
print(tok.convert_ids_to_tokens(enc["input_ids"][0]))
```

| Line | Meaning |
|------|---------|
| `enc["input_ids"]` | The **token ID numbers** for each piece |
| `convert_ids_to_tokens(...)` | Reverse: turns IDs back into readable tokens |

### Output
```
tensor([[101, 19204, 17629, 2015, 3975, 4678, 2616, 1012, 102]])
['[CLS]', 'token', '##izer', '##s', 'split', 'rare', 'words', '.', '[SEP]']
```

### Key Observations
| Token | Meaning |
|-------|---------|
| `[CLS]` (101) | **Start** marker BERT adds automatically |
| `[SEP]` (102) | **End/separator** marker |
| `token`, `##izer`, `##s` | Word **"Tokenizers" split into sub-words**! |
| `##` prefix | Means "attach to previous piece" (continuation) |

> The `##` splitting is how tokenizers handle rare words — break them into known pieces instead of failing.

---

## The Code — Part 2: Batch (Multiple Sentences)

```python
batch = tok(["short one", "a much longer sentence here"],
            padding=True, truncation=True, return_tensors="pt")
print(batch["input_ids"])
print(batch["attention_mask"])
```

| Parameter | Meaning |
|-----------|---------|
| `padding=True` | Pad shorter sentences with **0s** so all rows are the same length (tensors must be rectangular) |
| `truncation=True` | Cut sentences that are too long |

### Output
```
input_ids:
tensor([[101, 2460, 2028, 102,   0,   0,   0],    ← "short one" + padding
        [101, 1037, 2172, 2936, 6251, 2182, 102]]) ← longer sentence

attention_mask:
tensor([[1, 1, 1, 1, 0, 0, 0],    ← real tokens = 1, padding = 0
        [1, 1, 1, 1, 1, 1, 1]])
```

---

## What is `attention_mask`?

Tells the model **which tokens are real (1)** and **which are padding (0)** — so it **ignores the padding** during processing.

```
input_ids:      [101, 2460, 2028, 102,  0,  0,  0]
attention_mask: [  1,    1,    1,   1,  0,  0,  0]
                 └─── real tokens ───┘ └─padding─┘
```

> Without this mask, the model would waste attention on meaningless 0-padding.

---

## Key Takeaways

| Point | Detail |
|-------|--------|
| `AutoTokenizer` | Auto-loads correct tokenizer for any model |
| `input_ids` | Text → token ID numbers |
| `[CLS]` / `[SEP]` | Auto-added start/end markers (101/102) |
| `##` | Sub-word continuation (rare-word splitting) |
| `padding=True` | Makes batch rows equal length with 0s |
| `attention_mask` | 1 = real token, 0 = ignore (padding) |
| `return_tensors="pt"` | Output as PyTorch tensors |